# Founder-Departure OSS Truck-Factor Corpus

This notebook demonstrates the **assembly step** of the Founder-Departure OSS Truck-Factor Corpus dataset.

**Research question:** what determines whether an open-source project survives its founder stepping away?

The full pipeline mines GitHub repositories (`search_candidates.py`, `mine_repo.py`, `run_mining.py`) to compute, per repo and per year, the **Degree-of-Authorship (DOA)** metric and the **Truck Factor (TF)** from Avelino et al. (ICPC 2016), then flags a **Truck-Factor-Developer-Detachment (TFDD)** event: the first year the sole truck-factor developer (who is also the project's founder) has gone silent for >= 1 year. Repos with a qualifying founder-only TFDD plus >= 3 years of subsequent history are kept; the rest are discarded and logged with a reason.

This notebook runs the **final assembly script, `data.py`**, exactly as written, over a small subset of the raw per-repo mining outputs (`temp/repo_results/*.json`) that would normally be produced by the (expensive, GitHub-API-bound) mining scripts. It builds the same `input`/`output` example schema as the full corpus and visualizes the result.

## Setup

Install dependencies. `loguru` is not pre-installed on Colab; core scientific packages (numpy, pandas, matplotlib) are pinned to Colab's exact versions when running locally, and left untouched on Colab itself.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, matplotlib — pre-installed on Colab, install locally only to match Colab's env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
# Imports — same as the original data.py, plus matplotlib/pandas for the results section
import glob
import json
import sys
from pathlib import Path

from loguru import logger

import pandas as pd
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Load demo data

`mini_demo_data.json` is a curated subset of 25 raw per-repo mining results (15 `qualified`, 10 `discarded`) taken directly from `temp/repo_results/*.json` — the real intermediate files the full mining pipeline (`run_mining.py` / `mine_repo.py`) produces for each candidate repo, before `data.py` assembles them into the final dataset. Loading tries the GitHub-hosted copy first (for Colab), then falls back to the local file.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data)} raw repo-mining result files")
print("Example keys:", list(data.keys())[:5])

## Config

The original `data.py` has essentially one tunable knob: `RESULTS_GLOB`, the glob pattern selecting which mined per-repo result files to assemble. In the full pipeline this points at `temp/repo_results/*.json` (219 files: 32 qualified, 187 discarded). Here we start with the absolute minimum — a handful of files — and can scale up to the full `mini_demo_data.json` subset (25 files) in one step below, since that's all this demo ships with.

In [ ]:
# N_FILES: how many of the loaded raw mining-result files to assemble.
# Start at the minimum (all 25 shipped in mini_demo_data.json — already tiny), scale up here if a larger
# mini_demo_data.json is provided later.
N_FILES = len(data)  # minimum useful value = all files in the demo subset

## `build_example`: raw mined repo -> dataset example

This is `data.py`'s `build_example` function, copied as-is. It takes one raw mined repo-result dict (the pre-TFDD window, the TFDD-snapshot covariates, and the survival label) and turns it into one dataset example: `input` is a JSON string of the predictor covariates, `output` is the survival label, and everything else is carried as `metadata_*` fields for downstream verification without re-cloning the repo.

In [ ]:
def build_example(r: dict) -> dict:
    pre = r["pre_tfdd_window"]
    cov = r["tfdd_snapshot_covariates"]
    input_features = {
        "founder_commit_share_pre_tfdd": pre["founder_commit_share"],
        "n_distinct_new_primary_owners_pre_tfdd": pre["n_distinct_new_primary_owners"],
        "founder_early_authorship_share": r["founder_early_authorship_share"],
        "stars": cov["stars"],
        "forks": cov["forks"],
        "total_contributors": cov["total_contributors"],
        "language": cov["language"],
        "license": cov["license"],
        "project_age_days": cov["project_age_days"],
        "n_commits_total": r["n_commits"],
        "n_files_total": r["n_files"],
        "history_span_years": r["history_span_years"],
    }
    example = {
        "input": json.dumps(input_features, sort_keys=True),
        "output": r["survival_label"],
        "metadata_full_name": r["full_name"],
        "metadata_activity_bucket": r["activity_bucket"],
        "metadata_founder": r["founder"],
        "metadata_tfdd": r["tfdd"],
        "metadata_pre_tfdd_window": pre,
        "metadata_tfdd_snapshot_covariates": cov,
        "metadata_yearly_doa_tf_tables": r["yearly_tables"],
        "metadata_post_tfdd_monthly_commits": r["post_tfdd_monthly_commits"],
        "metadata_post_tfdd_months_available": r["post_tfdd_months_available"],
        "metadata_years_after_tfdd": r["years_after_tfdd"],
        "metadata_repo_meta": r["meta"],
        "metadata_repo_first_commit": r["repo_first_commit"],
        "metadata_repo_last_commit": r["repo_last_commit"],
        "metadata_task_type": "binary_classification",
        "metadata_n_classes": 2,
    }
    return example

## `main`: assemble the corpus

This mirrors `data.py`'s `main()` exactly, with one minimal change: instead of `glob.glob(RESULTS_GLOB)` + `Path(f).read_text()` reading files off disk, it iterates the raw-result dicts already loaded into `data` (the in-memory equivalent of the same `temp/repo_results/*.json` files), limited to `N_FILES` from the config cell above. The qualified/discarded split, discard-reason tally, and `build_example` call are untouched.

In [ ]:
files = list(data.items())[:N_FILES]
logger.info(f"Found {len(files)} mined repo result files")
qualified = []
discard_reasons = {}
for fname, r in files:
    if r.get("status") == "qualified":
        qualified.append(r)
    else:
        reason = r.get("discard_reason", "unknown")
        discard_reasons[reason] = discard_reasons.get(reason, 0) + 1
logger.info(f"Qualified repos: {len(qualified)}")
logger.info(f"Discard reasons: {json.dumps(discard_reasons, indent=2)}")

examples = [build_example(r) for r in qualified]
output = {
    "metadata": {
        "source": "GitHub REST search API (candidate discovery) + git log (--filter=blob:none) "
                   "for full commit history mining",
        "description": "Single-founder GitHub repos with founder-only Truck-Factor-Developer-"
                        "Detachment (TFDD) events, per Avelino et al. ICPC'16 (DOA/TF algorithm) "
                        "and Avelino et al. ESEM'19 (TFDD/survival definitions). Each example is "
                        "one qualifying repo; input=pre-TFDD/snapshot covariates, output=survival "
                        "label (Active_survived / Inactive_did_not_survive).",
        "n_qualified": len(qualified),
        "discard_reason_counts": discard_reasons,
        "doa_formula": "DOA(d,f) = 3.293 + 1.098*FirstAuthor(d,f) + 0.164*Deliveries(d,f) "
                       "- 0.321*ln(1+Acceptances(d,f))",
        "tf_algorithm": "greedy removal of highest-file-count DOA-primary-author while "
                        "remaining-authors' file coverage >= 0.5",
    },
    "datasets": [
        {"dataset": "founder_departure_tfdd_corpus", "examples": examples}
    ],
}
logger.info(f"Assembled {len(examples)} examples into the corpus (in-memory, not written to disk in this demo)")

## Results

A look at the assembled examples: the survival-label distribution, the discard-reason breakdown, and the predictor covariates (`input`) for the qualified repos.

In [ ]:
df = pd.DataFrame([
    {"full_name": ex["metadata_full_name"], "survival_label": ex["output"],
     **json.loads(ex["input"])}
    for ex in examples
])
print(f"{len(examples)} qualified examples assembled\n")
print(df[["full_name", "survival_label", "language", "stars", "founder_commit_share_pre_tfdd",
          "n_distinct_new_primary_owners_pre_tfdd"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df["survival_label"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Survival label distribution (qualified repos)")
axes[0].set_ylabel("count")
axes[0].tick_params(axis="x", rotation=20)

pd.Series(discard_reasons).sort_values().plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Discard reasons (non-qualifying repos)")
axes[1].set_xlabel("count")

plt.tight_layout()
plt.show()